<a href="https://colab.research.google.com/github/Metropoliya/AI/blob/main/SeamlessM4T.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SeamlessM4T

| Code Credits | Link |
| ----------- | ---- |
| 🎉 seamless_communication | [![GitHub Repository](https://img.shields.io/github/stars/facebookresearch/seamless_communication?style=social)](https://github.com/facebookresearch/seamless_communication) |
| 🚀 Online inference | [![Hugging Face Spaces](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-Spaces-blue)](https://huggingface.co/spaces/facebook/seamless_m4t) |
| 🔥 Discover More Colab Notebooks | [![GitHub Repository](https://img.shields.io/badge/GitHub-Repository-black?style=flat-square&logo=github)](https://github.com/R3gm/InsightSolver-Colab/) |


SeamlessM4T is a groundbreaking Massively Multilingual & Multimodal Machine Translation model, bridging speech and text translation for up to 100 languages.

Generally, the translation from one audio to another is done with several intermediate steps, such as transcription, translation, and later conversion to audio (Cascaded systems), as in [SoniTranslate](https://github.com/R3gm/SoniTranslate). However, the current model allows us to perform all these tasks.

Utility Functions and Libraries

In [1]:
# 1. Устанавливаем только стабильные библиотеки (без fairseq2!)
!pip install -q transformers accelerate pydub torchaudio scipy

import os
import torch
import torchaudio
from transformers import AutoProcessor, SeamlessM4Tv2Model
from pydub import AudioSegment
from pydub.silence import split_on_silence
import scipy.io.wavfile

# 2. Настройки
my_file = "/content/026-03-18_23-54-17.mp3" # Ваш файл
device = "cuda:0" if torch.cuda.is_available() else "cpu"

print("Загружаем модель SeamlessM4T v2 (это займет пару минут)...")
# Используем стабильную интеграцию от Hugging Face
processor = AutoProcessor.from_pretrained("facebook/seamless-m4t-v2-large")
model = SeamlessM4Tv2Model.from_pretrained("facebook/seamless-m4t-v2-large").to(device)

# 3. Нарезаем 35 минут на удобные фразы по паузам
print("Нарезаем аудио на фразы...")
audio = AudioSegment.from_mp3(my_file)
chunks = split_on_silence(
    audio,
    min_silence_len=600,
    silence_thresh=audio.dBFS-16,
    keep_silence=300
)

os.makedirs("/content/translated_audio", exist_ok=True)
print(f"Успех! Получилось {len(chunks)} кусочков. Начинаем перевод!")

# 4. Магия перевода: Русский голос -> Английский голос
for i, chunk in enumerate(chunks):
    temp_wav = f"/content/temp_chunk_{i}.wav"
    # Модель лучше всего "слышит" аудио с частотой 16000 Гц
    chunk.set_frame_rate(16000).export(temp_wav, format="wav")

    print(f"Перевожу часть {i+1} из {len(chunks)}...")

    try:
        # Загружаем звук в понятном для нейросети формате
        audio_tensor, orig_freq = torchaudio.load(temp_wav)
        audio_inputs = processor(audios=audio_tensor, return_tensors="pt", sampling_rate=16000).to(device)

        # Генерируем английскую речь (tgt_lang="eng")
        audio_array = model.generate(**audio_inputs, tgt_lang="eng")[0].cpu().numpy().squeeze()

        # Сохраняем готовую переведенную фразу
        out_file = f'/content/translated_audio/english_chunk_{i}.wav'
        scipy.io.wavfile.write(out_file, rate=model.config.sampling_rate, data=audio_array)

    except Exception as e:
        print(f"Произошла ошибка на кусочке {i+1}: {e}")

print("Всё готово! Ищите английские файлы в папке /content/translated_audio/")

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


Загружаем модель SeamlessM4T v2 (это займет пару минут)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

Instantiating a decoder SeamlessM4Tv2Attention without passing `layer_idx` is not recommended and will lead to errors during the forward call, if caching is used. Please make sure to provide a `layer_idx` when creating this class.


Loading weights:   0%|          | 0/2232 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

Нарезаем аудио на фразы...


FileNotFoundError: [Errno 2] No such file or directory: '/content/026-03-18_23-54-17.mp3'

Load the model

In [ ]:
import os
from pydub import AudioSegment
from google.colab import files

folder_path = "/content/translated_audio/"
print("Собираем все английские кусочки воедино...")

# Ищем все файлы, которые начинаются на 'english_chunk_'
chunk_files = [f for f in os.listdir(folder_path) if f.startswith("english_chunk_") and f.endswith(".wav")]

# Очень важный шаг: сортируем файлы по их числовому номеру
chunk_files.sort(key=lambda x: int(x.split('_')[2].split('.')[0]))

# Создаем пустой аудио-трек
final_audio = AudioSegment.empty()

# Проходимся по списку и склеиваем
for chunk_name in chunk_files:
    chunk_path = os.path.join(folder_path, chunk_name)
    audio_chunk = AudioSegment.from_wav(chunk_path)

    # Прибавляем фразу к общему треку
    final_audio += audio_chunk

    # Добавляем 0.3 секунды тишины между фразами, чтобы речь не сливалась в кашу
    final_audio += AudioSegment.silent(duration=300)

# Сохраняем финальный результат
output_filename = "/content/final_english_track.mp3"
print("Сохраняем итоговый MP3 файл...")
final_audio.export(output_filename, format="mp3")

print("Всё готово! Запускаю скачивание...")
# Автоматически скачиваем файл из Colab на ваш компьютер
files.download(output_filename)